# Scaling beyond a single computer

The same circuit and the same code can run on CESGA's cluster by changing only the value of `infrastructure`. Most readers following this series will not have access to a cluster with SLURM or to real quantum hardware: that is why the examples in this notebook are limited to argument validation, which does happen without needing that access, instead of an actual run.

## CUNQA: distributed and simulated QPUs

`infrastructure="cunqa"` runs against CESGA's CUNQA platform, which spreads simulated QPUs across several nodes managed by SLURM. The `cunqa` package needs to be installed separately, it does not come bundled with `polypus-quantum`; see [Installation](https://github.com/Bahia-Software/polypus#installation) in the main README.

Three additional keyword arguments size the SLURM reservation, with an effect only on this infrastructure:

- `nodes`: number of SLURM nodes to reserve. Defaults to 1.
- `cores_per_qpu`: CPU cores per QPU. Defaults to 2.
- `walltime`: SLURM walltime, the maximum time the QPUs stay reserved, as `HH:MM:SS`. Defaults to `"10:00:00"`.

A real call would look like this:

```python
result = polypus.run_quantum_circuit(
    qc,
    shots=1000,
    infrastructure="cunqa",
    n_qpus=10,
    nodes=2,
    cores_per_qpu=4,
    walltime="02:00:00",
)
```

`nodes` and `cores_per_qpu` must be at least 1, and this is checked before ever reaching SLURM, so the error shows up without having CUNQA installed:

In [ ]:
import polypus

bell = polypus.Circuit(2).h(0).cx(0, 1).measure_all()

try:
    polypus.run_quantum_circuit(bell, shots=1000, infrastructure="cunqa", nodes=0)
except ValueError as e:
    print(e)

`walltime` only accepts the `HH:MM:SS` form: hours with one or more digits, so `"48:00:00"` is valid, and minutes and seconds with two digits below 60. SLURM's form with days, `D-HH:MM:SS`, is rejected, also before reaching SLURM:

In [ ]:
try:
    polypus.run_quantum_circuit(
        bell, shots=1000, infrastructure="cunqa", walltime="2-00:00:00"
    )
except ValueError as e:
    print(e)

When the walltime runs out, the QPUs are released, so for a long training it should be set above the expected duration. Polypus helps with that: when training on CUNQA, it logs a warning after the first generation if, at that pace, the run would outlast the walltime.

CUNQA's QPUs simulate with Aer, so with `infrastructure="cunqa"` the `backend` must be Aer, the default `"aer"`. Any other value is rejected instead of being silently ignored:

In [ ]:
try:
    polypus.run_quantum_circuit(
        bell, shots=1000, infrastructure="cunqa", backend="polypus"
    )
except ValueError as e:
    print(e)

A `noise_model`, a Qiskit Aer description of the errors of a real machine that the local Aer backend can simulate, is not accepted either: CUNQA's QPUs are configured when they are reserved, not on every run:

In [ ]:
from qiskit_aer.noise import NoiseModel

try:
    polypus.run_quantum_circuit(
        bell, shots=1000, infrastructure="cunqa", noise_model=NoiseModel()
    )
except ValueError as e:
    print(e)

## QMIO: real quantum hardware

`infrastructure="qmio"` runs directly against QMIO, CESGA's real quantum computer, through a pure-Rust client. The package published on PyPI does not include this backend: Polypus needs to be built from source with the `qmio` feature enabled, so the attempt fails with a clear message:

In [ ]:
try:
    polypus.run_quantum_circuit(bell, shots=1000, infrastructure="qmio")
except ValueError as e:
    print(e)

Being real hardware, QMIO also does not accept an explicit seed: the noise of the machine itself cannot be fixed like that of a simulator, and `seed` gets rejected just as quickly:

In [ ]:
try:
    polypus.run_quantum_circuit(bell, shots=1000, infrastructure="qmio", seed=42)
except ValueError as e:
    print(e)

A source build with the `qmio` feature also needs to know where QMIO is: there is no default address. It is read from the `ZMQ_SERVER` environment variable, or passed with `options={"endpoint": ...}`, and without either the run fails with a `polypus.QmioError`.

## Summary

This notebook covered how the same code targets CUNQA or QMIO by changing `infrastructure`, each one's additional arguments, and their real validations, seen without needing access to a cluster or to quantum hardware.

The next section returns to concrete circuits: quantum teleportation and the quantum Fourier transform.

## Next step

Continue with: [`04a_teleportation.ipynb`](04a_teleportation.ipynb).